<a href="https://colab.research.google.com/github/MarioRojasV/CNN_Models_Comparison_Bird_Classification/blob/main/scr/mobilenetv2_model/MobileNetV2_Birds_COMPDES_2026_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os

print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

os.listdir('/content/drive/MyDrive')

Mounted at /content/drive


['Universidad', 'Colab Notebooks']

In [ ]:
BASE_PATH = "/content/drive/MyDrive/Universidad/COMPDES2026_Aves_CR"

DATASET_PATH = BASE_PATH + "/Dataset"

test_path = DATASET_PATH + "/Testing_set/Regular_photos"

print(test_path)

os.listdir(DATASET_PATH)

os.listdir(test_path)

/content/drive/MyDrive/Universidad/COMPDES2026_Aves_CR/Dataset/Testing_set/Regular_photos


['Quiscalus_mexicanus_testing_photos', 'Turdus_grayi_testing_photos']

In [ ]:
model = tf.keras.models.load_model(
    "/content/drive/MyDrive/Universidad/COMPDES2026_Aves_CR/Resultados_MobileNetV2/MobileNetV2_baseline.keras"
)

model.summary()

Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_13 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         2,562 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,265,672 (8.64 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 5,126 (20.03 KB)

In [ ]:
BATCH_SIZE = 32

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(224,224),
    batch_size=BATCH_SIZE,
    pad_to_aspect_ratio=True
)

Found 80 files belonging to 2 classes.


In [ ]:
class_names = test_ds.class_names

print(class_names)

['Quiscalus_mexicanus_testing_photos', 'Turdus_grayi_testing_photos']


In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input


test_ds = test_ds.map(
    lambda x, y: (preprocess_input(x), y)
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print("Test accuracy:", test_accuracy)
print("Test loss:", test_loss)

3/3 ━━━━━━━━━━━━━━━━━━━━ 37s 6s/step - accuracy: 0.9625 - loss: 0.1502
Test accuracy: 0.9624999761581421
Test loss: 0.15018348395824432


In [ ]:
y_true = []
y_pred = []

In [ ]:
for images, labels in test_ds:

    predictions = model.predict(images)

    predicted_classes = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step


In [ ]:
y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
print(y_true)
print(y_pred)

[1 0 1 1 0 0 0 0 0 1 0 1 1 1 0 0 1 0 1 1 1 1 0 1 1 1 0 0 0 0 1 1 1 1 1 0 0
 0 1 1 0 0 0 1 1 1 0 1 0 0 0 0 1 0 0 1 0 1 0 0 0 0 0 1 1 1 1 1 1 0 0 0 1 0
 1 1 1 0 1 0]
[1 0 1 1 0 0 0 0 0 1 0 1 1 1 0 1 1 0 1 1 1 1 0 1 1 1 0 0 0 0 1 1 1 1 1 0 0
 0 1 1 0 0 1 1 1 1 0 1 0 0 0 0 1 0 0 1 0 1 0 0 0 0 0 1 1 1 1 1 1 0 0 0 1 1
 1 1 1 0 1 0]


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_true,
    y_pred
)

print(cm)

[[37  3]
 [ 0 40]]


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )
)

                                    precision    recall  f1-score   support

Quiscalus_mexicanus_testing_photos       1.00      0.93      0.96        40
       Turdus_grayi_testing_photos       0.93      1.00      0.96        40

                          accuracy                           0.96        80
                         macro avg       0.97      0.96      0.96        80
                      weighted avg       0.97      0.96      0.96        80

